# Dataset Versioning with MLflow

## Topic Goal


A model is not only connected to code and parameters. It is also strongly connected to the exact dataset used for training.

This notebook demonstrates:

1. Dataset path tracking
2. Dataset version name
3. Dataset hash/fingerprint
4. Dataset row count
5. Dataset feature count
6. Dataset column schema
7. Feature list tracking
8. Target distribution tracking
9. Baseline vs current dataset comparison
10. Dataset metadata artifact generation
11. Same-run model logging with input example and signature
12. Model registration using the same-run `model_uri`

This is useful for reproducibility, audit, governance, and debugging model performance changes.


## 1. Import Libraries

We import MLflow, pandas, scikit-learn, and helper libraries.

We use:

- `hashlib` to create dataset fingerprint
- `json` to save dataset metadata
- MLflow to log dataset metadata, model, signature, and registry information


In [ ]:
# Import os to create folders and manage local paths.
import os

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import json to save dataset metadata as JSON.
import json

# Import hashlib to create dataset hash/fingerprint.
import hashlib

# Import datetime to record dataset metadata creation time.
from datetime import datetime

# Import MLflow for tracking, model logging, and registry.
import mlflow

# Import MLflow scikit-learn flavor for logging sklearn pipelines.
import mlflow.sklearn

# Import infer_signature to capture model input/output schema.
from mlflow.models.signature import infer_signature

# Import pandas for reading and processing CSV data.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import train_test_split for splitting data.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for separate preprocessing of feature groups.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical columns.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical columns.
from sklearn.preprocessing import StandardScaler

# Import Pipeline to combine preprocessing and model.
from sklearn.pipeline import Pipeline

# Import RandomForestClassifier for churn prediction.
from sklearn.ensemble import RandomForestClassifier

# Import evaluation metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Configure MLflow and Local Folders

This notebook uses local file-based MLflow tracking.

The dataset versioning metadata is saved in `outputs/` and also logged to MLflow.


In [ ]:
# Define experiment name.
EXPERIMENT_NAME = "advanced_13_dataset_versioning"

# Define run name.
RUN_NAME = "13_dataset_versioning_same_run_usecase"

# Remove inherited MLflow Project experiment ID if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run ID if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for dataset metadata artifacts.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for supporting files.
os.makedirs("artifacts", exist_ok=True)

# Configure local MLflow tracking.
mlflow.set_tracking_uri("file:./mlruns")

# Define baseline training dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Define optional current dataset path for comparison.
CURRENT_DATA_PATH = "datasets/customer_churn_current_500.csv"

# Define dataset version name.
DATASET_VERSION = "v1_baseline_customer_churn"

# Define current dataset version name.
CURRENT_DATASET_VERSION = "v2_current_customer_churn"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Print setup information.
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Training dataset:", DATA_PATH)
print("Dataset version:", DATASET_VERSION)


## 3. Helper Function to Create Dataset Hash

A dataset hash is a fingerprint of the file contents.

If the dataset changes, the hash will also change.

This helps identify exactly which data file was used during training.


In [ ]:
# Define helper function to create MD5 hash for a file.
def calculate_file_hash(file_path):
    # Create hash object.
    hash_object = hashlib.md5()

    # Open file in binary mode.
    with open(file_path, "rb") as file:
        # Read file in chunks to support larger files.
        for chunk in iter(lambda: file.read(4096), b""):
            # Update hash with each chunk.
            hash_object.update(chunk)

    # Return hexadecimal hash string.
    return hash_object.hexdigest()


## 4. Load Baseline Dataset and Capture Version Metadata

We capture dataset metadata before training.

This includes:

- dataset path
- dataset version
- hash
- row count
- column count
- feature count
- target column
- categorical columns
- numerical columns
- schema
- feature list


In [ ]:
# Check whether baseline dataset exists.
if not os.path.exists(DATA_PATH):
    # Raise clear error if dataset is missing.
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# Load baseline dataset.
df = pd.read_csv(DATA_PATH)

# Display first five rows.
display(df.head())

# Define target column.
target_column = "churn"

# Create feature matrix.
X = df.drop(columns=["customer_id", target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Calculate dataset hash.
dataset_hash_md5 = calculate_file_hash(DATA_PATH)

# Calculate row count.
row_count = len(df)

# Calculate column count.
column_count = len(df.columns)

# Calculate feature count.
feature_count = X.shape[1]

# Create feature list.
feature_columns = X.columns.tolist()

# Capture pandas dtypes as schema.
dataset_schema = {column: str(dtype) for column, dtype in df.dtypes.items()}

# Calculate target distribution.
target_distribution = df[target_column].value_counts(dropna=False).to_dict()

# Print core metadata.
print("Dataset hash:", dataset_hash_md5)
print("Rows:", row_count)
print("Columns:", column_count)
print("Features:", feature_count)
print("Target distribution:", target_distribution)


## 5. Optional: Compare Baseline Dataset with Current Dataset

A production team may compare a baseline training dataset with a newer/current dataset.

This is not full drift detection, but it helps show whether row counts, schema, or selected feature means have changed.


In [ ]:
# Initialize current dataset metadata.
current_dataset_metadata = None

# Check whether current dataset exists.
if os.path.exists(CURRENT_DATA_PATH):
    # Load current dataset.
    current_df = pd.read_csv(CURRENT_DATA_PATH)

    # Calculate current dataset hash.
    current_dataset_hash_md5 = calculate_file_hash(CURRENT_DATA_PATH)

    # Create current metadata.
    current_dataset_metadata = {
        "dataset_path": CURRENT_DATA_PATH,
        "dataset_version": CURRENT_DATASET_VERSION,
        "dataset_hash_md5": current_dataset_hash_md5,
        "row_count": len(current_df),
        "column_count": len(current_df.columns),
        "schema": {column: str(dtype) for column, dtype in current_df.dtypes.items()},
    }

    # Create numeric mean comparison for common numeric columns.
    numeric_comparison = pd.DataFrame({
        "feature": numerical_columns,
        "baseline_mean": [df[col].mean() for col in numerical_columns],
        "current_mean": [current_df[col].mean() for col in numerical_columns],
    })

    # Calculate percentage change.
    numeric_comparison["percent_change"] = (
        (numeric_comparison["current_mean"] - numeric_comparison["baseline_mean"])
        / numeric_comparison["baseline_mean"]
    ) * 100

    # Save comparison.
    numeric_comparison.to_csv("outputs/baseline_vs_current_numeric_comparison.csv", index=False)

    # Display comparison.
    display(numeric_comparison)

else:
    # Print message if current dataset is unavailable.
    print("Current dataset not found. Skipping baseline vs current comparison.")


## 6. Save Dataset Metadata Artifacts

Dataset versioning metadata should be stored as artifacts.

This allows future users to inspect the dataset metadata used to train the model.


In [ ]:
# Create dataset metadata dictionary.
dataset_metadata = {
    "metadata_created_at": datetime.now().isoformat(),
    "dataset_path": DATA_PATH,
    "dataset_version": DATASET_VERSION,
    "dataset_hash_md5": dataset_hash_md5,
    "row_count": row_count,
    "column_count": column_count,
    "feature_count": feature_count,
    "target_column": target_column,
    "feature_columns": feature_columns,
    "categorical_columns": categorical_columns,
    "numerical_columns": numerical_columns,
    "schema": dataset_schema,
    "target_distribution": target_distribution,
    "current_dataset_metadata": current_dataset_metadata,
}

# Save metadata as JSON.
with open("outputs/dataset_metadata.json", "w") as file:
    json.dump(dataset_metadata, file, indent=4)

# Save feature list as text.
with open("outputs/feature_list.txt", "w") as file:
    for feature in feature_columns:
        file.write(feature + "\n")

# Save schema as JSON.
with open("outputs/dataset_schema.json", "w") as file:
    json.dump(dataset_schema, file, indent=4)

# Save target distribution as JSON.
with open("outputs/target_distribution.json", "w") as file:
    json.dump(target_distribution, file, indent=4)

# Print saved artifact names.
print("Saved dataset metadata artifacts:")
print("- outputs/dataset_metadata.json")
print("- outputs/feature_list.txt")
print("- outputs/dataset_schema.json")
print("- outputs/target_distribution.json")


## 7. Prepare Data for Model Training

Now we build preprocessing logic and split the dataset.

The dataset metadata captured above will be logged together with the model run.


In [ ]:
# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Standardize numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print data split information.
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 8. Train and Evaluate Model

The model performance is now connected to the dataset version metadata.

If the data changes later, the metadata and hash help identify which dataset produced which result.


In [ ]:
# Create Random Forest classifier.
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    random_state=42
)

# Create complete pipeline with preprocessing and model.
pipeline = Pipeline(steps=[
    # Apply preprocessing first.
    ("preprocessor", preprocessor),

    # Train model after preprocessing.
    ("model", model),
])

# Train pipeline.
pipeline.fit(X_train, y_train)

# Generate test predictions.
predictions = pipeline.predict(X_test)

# Calculate accuracy.
accuracy = accuracy_score(y_test, predictions)

# Calculate precision.
precision = precision_score(y_test, predictions, zero_division=0)

# Calculate recall.
recall = recall_score(y_test, predictions, zero_division=0)

# Calculate F1-score.
f1 = f1_score(y_test, predictions, zero_division=0)

# Create classification report.
report = classification_report(y_test, predictions, zero_division=0)

# Save classification report.
with open("outputs/classification_report.txt", "w") as file:
    file.write(report)

# Print metrics.
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print(report)


## 9. Infer Model Signature

The model signature documents the model input and output schema.

This is separate from dataset schema, but both are useful:

- Dataset schema describes the source data file.
- Model signature describes the model input/output contract.


In [ ]:
# Create input example from test data.
input_example = X_test.head(5)

# Generate sample predictions for signature.
sample_predictions = pipeline.predict(input_example)

# Infer model signature.
signature = infer_signature(input_example, sample_predictions)

# Display input example.
display(input_example)

# Print sample predictions.
print("Sample predictions:", sample_predictions)


## 10. Same-Run MLflow Logging with Dataset Versioning

This is the main production-style run.

Inside the **same MLflow run**, we log:

- dataset version
- dataset hash
- row count
- column count
- feature count
- target column
- feature list
- schema artifact
- metadata artifact
- model metrics
- model with signature and input example
- model URI

Then we register the model using the same `model_uri`.


In [ ]:
# Start the SAME MLflow run for dataset metadata, metrics, model, signature, and model URI.
with mlflow.start_run(run_name=RUN_NAME):

    # Log dataset path.
    mlflow.log_param("dataset_path", DATA_PATH)

    # Log dataset version name.
    mlflow.log_param("dataset_version", DATASET_VERSION)

    # Log dataset hash/fingerprint.
    mlflow.log_param("dataset_hash_md5", dataset_hash_md5)

    # Log row count.
    mlflow.log_param("row_count", row_count)

    # Log column count.
    mlflow.log_param("column_count", column_count)

    # Log feature count.
    mlflow.log_param("feature_count", feature_count)

    # Log target column.
    mlflow.log_param("target_column", target_column)

    # Log categorical column count.
    mlflow.log_param("categorical_column_count", len(categorical_columns))

    # Log numerical column count.
    mlflow.log_param("numerical_column_count", len(numerical_columns))

    # Log feature names as a comma-separated string.
    mlflow.log_param("feature_columns", ",".join(feature_columns))

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Log topic name.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Set dataset hash as a searchable tag.
    mlflow.set_tag("dataset_hash_md5", dataset_hash_md5)

    # Set dataset version as a searchable tag.
    mlflow.set_tag("dataset_version", DATASET_VERSION)

    # Set data source type.
    mlflow.set_tag("data_source_type", "csv")

    # Log accuracy.
    mlflow.log_metric("accuracy", accuracy)

    # Log precision.
    mlflow.log_metric("precision", precision)

    # Log recall.
    mlflow.log_metric("recall", recall)

    # Log F1-score.
    mlflow.log_metric("f1_score", f1)

    # Log dataset metadata artifacts.
    mlflow.log_artifact("outputs/dataset_metadata.json")
    mlflow.log_artifact("outputs/feature_list.txt")
    mlflow.log_artifact("outputs/dataset_schema.json")
    mlflow.log_artifact("outputs/target_distribution.json")
    mlflow.log_artifact("outputs/classification_report.txt")

    # Log baseline vs current comparison artifact if it exists.
    if os.path.exists("outputs/baseline_vs_current_numeric_comparison.csv"):
        mlflow.log_artifact("outputs/baseline_vs_current_numeric_comparison.csv")

    # Log model with signature and input example in the SAME run.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Register model using the same-run model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print model and registry details.
print("Same-run model URI:", model_uri)
print("Registered model name:", registered_model.name)
print("Registered model version:", registered_model.version)


## 11. Verify Dataset Metadata

This section reads the saved metadata back so students can see what was captured.


In [ ]:
# Read saved dataset metadata.
with open("outputs/dataset_metadata.json", "r") as file:
    saved_metadata = json.load(file)

# Print selected metadata.
print("Dataset version:", saved_metadata["dataset_version"])
print("Dataset hash:", saved_metadata["dataset_hash_md5"])
print("Row count:", saved_metadata["row_count"])
print("Feature count:", saved_metadata["feature_count"])
print("Target column:", saved_metadata["target_column"])

# Display feature list.
print("Feature columns:")
for feature in saved_metadata["feature_columns"]:
    print("-", feature)




- Dataset versioning tells us exactly which data was used to train the model.
- A dataset hash is like a fingerprint of the dataset file.
- If the dataset changes, the hash changes.
- Row count, feature count, schema, and target distribution help with reproducibility.
- Dataset metadata should be logged directly as MLflow parameters and also stored as artifacts.
- The model performance should always be interpreted together with the dataset version.
- In production, dataset versioning is often handled using tools such as DVC, LakeFS, Delta Lake, or data warehouse snapshots.
- MLflow can still log dataset metadata to create an audit trail.
